### Defining parameters

In [ ]:
import pandas as pd

year = 2024
columns_to_keep = [
    "locatie_code",
    "grootheid_code",
    "parameter_code",
    "hoedanigheid_code",
    "eventdatum",
    "event_aquocode",
    "eenheid_code",
]
data_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}.xlsx"
mapping_location_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\locations-mapped.xlsx"
mapping_substances_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\parameter_mapping_final.xlsx"

df_data = pd.read_excel(data_path,sheet_name=str(year))
df_mapping_location = pd.read_excel(mapping_location_path)
df_mapping_substances = pd.read_excel(mapping_substances_path)

### Defining functions

In [ ]:
def mapping_location(df_data, df_mapping):
    # Mapping column names
    original_loc_col = "locatie_code"
    target_loc_col = "Identitication unique de la station"

    # Keep one mapping row per location code
    df_mapping = df_mapping[[original_loc_col, target_loc_col]]

    # Merge station unique ID into df_data on locatie_code
    df_data = df_data.merge(df_mapping, on=original_loc_col, how="left")

    missing_count = df_data[target_loc_col].isna().sum()
    print(f"Missing location data: {missing_count} rows")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def mapping_substances(df_data, df_mapping):

    # ds column and equivalent mapping key
    original_parameter_col = "parameter_code"

    # Rename "wadar_PARCode" to "parameter_code"
    df_mapping = df_mapping.rename(columns={"wadar_PARCode": original_parameter_col})

    # Column you want to bring from mapping table
    target_parameter_col = "Unieke identificatie gemeten parameter"  

    df_mapping = df_mapping[[original_parameter_col, target_parameter_col]]
    print(f"Shape of df_mapping: {df_mapping.shape}")

    # # This keeps only rows where parameter exists in the mapping parameter table 
    # df_data = df_data.merge(df_mapping, on=original_parameter_col, how="inner")
    # This keeps all rows and adds the parameter column from the mapping table
    df_data = df_data.merge(df_mapping, on=original_parameter_col, how="left")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def drop_rows_without_parameter(df_data):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    df_data = df_data[df_data[target_parameter_col].notna()]
    print(f"Shape of df_data after dropping rows without parameter: {df_data.shape}")
    return df_data

def filter_data(df_data):
    #Removing df_data where waardebewerkings_methode_code is BER then print shape of filtered DataFrame
    filtered_df = df_data[df_data["waardebewerkings_methode_code"] != "BER"]
    print("Shape of filtered DataFrame (excluding BER):", filtered_df.shape)

    # Remove rows where bemonsteringshoogte_code is different -100 then print shape of filtered DataFrame
    filtered_df = filtered_df[filtered_df["bemonsteringshoogte_code"] == -100]
    print("Shape of filtered DataFrame (bemonsteringshoogte_code == -100):", filtered_df.shape)

    # Remove rwos where event_aquocode is differente to 0, 3, 90, or 99 and print shape of filtered DataFrame
    valid_aquocodes = [0, 3, 90, 99]
    filtered_df = filtered_df[filtered_df["event_aquocode"].isin(valid_aquocodes)]
    print("Shape of filtered DataFrame (valid event_aquocode):", filtered_df.shape)
    return filtered_df

def get_count_unique_values(filtered_df):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    # Get count of unique values in the target column after merge
    unique_values_after_merge = filtered_df[target_parameter_col].value_counts(dropna=False)
    print(unique_values_after_merge)
    print(f"Shape of unique_values_after_merge: {unique_values_after_merge.shape}")
    return unique_values_after_merge

def check_duplicates(filtered_df):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    # Keep only the columns of interest for duplicate checking
    columns_to_keep = [target_loc_col ,"grootheid_code",target_parameter_col , "hoedanigheid_code", "eventdatum", "event_aquocode", "eenheid_code"]
    filtered_df_duplicates = filtered_df[columns_to_keep].copy()
    filtered_df_duplicates 

    # Check if there are duplicated rows in the filtered DataFrame
    has_identical_rows = filtered_df_duplicates.duplicated().any()
    print("Identical rows found:" if has_identical_rows else "No identical rows found.")

def split_by_station(df_data):
    target_loc_col = "Identitication unique de la station"
    dfs_by_station = {
        station_id: group_df.reset_index(drop=True)
        for station_id, group_df in df_data.groupby(target_loc_col, dropna=False)
    }
    print(f"Created {len(dfs_by_station)} station DataFrames")
    return dfs_by_station

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    columns_to_keep = [
        target_loc_col,
        "grootheid_code",
        target_parameter_col,
        "hoedanigheid_code",
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)
            .agg(
                n_measurements=("eventdatum", "size"),
                n_unique_dates=("eventdatum", "nunique"),
                first_date=("eventdatum", "min"),
                last_date=("eventdatum", "max"),
            )
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def print_repeated_cases_shapes(repeated_cases_by_station):
    total_rows = 0

    for station_id, station_df in repeated_cases_by_station.items():
        n_rows = station_df.shape[0]
        total_rows += n_rows
        print(f"{station_id}: {station_df.shape} ({n_rows} rows)")

    print(f"Total stations: {len(repeated_cases_by_station)}")
    print(f"Total rows: {total_rows}")

def print_rows_per_event_aquocode(repeated_cases_by_station, valid_aquocodes=None):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    total_invalid_rows = 0

    for station_id, station_df in repeated_cases_by_station.items():
        counts = station_df["event_aquocode"].value_counts(dropna=False).sort_index()
        invalid_mask = ~station_df["event_aquocode"].isin(valid_aquocodes)
        n_invalid = invalid_mask.sum()

        print(f"\n{station_id}")
        print(counts)

        if n_invalid > 0:
            total_invalid_rows += n_invalid
            invalid_counts = station_df.loc[invalid_mask, "event_aquocode"].value_counts(dropna=False).sort_index()
            print(f"  Invalid event_aquocode rows: {n_invalid}")
            print(f"  Invalid codes: {invalid_counts.to_dict()}")
        else:
            print("  All rows have valid event_aquocode")

    print(f"\nTotal invalid rows across all stations: {total_invalid_rows}")

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    grand_total_cases = 0

    for station_id, station_df in repeated_cases_by_station.items():
        print(f"\n{station_id}")

        station_total_cases = 0

        for event_aquocode in valid_aquocodes:
            df = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df)
            n_unique_parameters = df[target_parameter_col].nunique()
            station_total_cases += n_cases

            print(
                f"  aquocode {event_aquocode}: "
                f"{n_cases} cases, {n_unique_parameters} unique parameters"
            )

            if n_cases > 0:
                print(df[target_parameter_col].value_counts(ascending=False))

        print(f"  station total: {station_total_cases}")
        grand_total_cases += station_total_cases

    print(f"\nTotal cases across all stations: {grand_total_cases}")

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    hoedanigheid_col = "hoedanigheid_code"
    grand_total_cases = 0

    for station_id, station_df in repeated_cases_by_station.items():
        print(f"\n{station_id}")

        station_total_cases = 0

        for event_aquocode in valid_aquocodes:
            df_aquocode = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df_aquocode)
            n_unique_parameters = df_aquocode[target_parameter_col].nunique()
            station_total_cases += n_cases

            print(
                f"  aquocode {event_aquocode}: "
                f"{n_cases} cases, {n_unique_parameters} unique parameters"
            )

            if n_cases > 0:
                print("    all hoedanigheid_code:")
                print(df_aquocode[target_parameter_col].value_counts(ascending=False))

            # Extra breakdown: per hoedanigheid_code
            for hoedanigheid_code, df_hoedanigheid in df_aquocode.groupby(
                hoedanigheid_col, dropna=False
            ):
                n_cases_h = len(df_hoedanigheid)
                n_unique_parameters_h = df_hoedanigheid[target_parameter_col].nunique()

                print(
                    f"    hoedanigheid_code {hoedanigheid_code}: "
                    f"{n_cases_h} cases, {n_unique_parameters_h} unique parameters"
                )

                if n_cases_h > 0:
                    print(df_hoedanigheid[target_parameter_col].value_counts(ascending=False))

        print(f"  station total: {station_total_cases}")
        grand_total_cases += station_total_cases

    print(f"\nTotal cases across all stations: {grand_total_cases}")

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
    show_only_duplicates=True,
    top_n=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    hoedanigheid_col = "hoedanigheid_code"
    grand_total_cases = 0

    def print_parameter_summary(df, indent="      "):
        counts = df[target_parameter_col].value_counts(ascending=False)

        if show_only_duplicates:
            counts = counts[counts > 1]

        if top_n is not None:
            counts = counts.head(top_n)

        if len(counts) == 0:
            print(f"{indent}No duplicate parameters")
            return

        for parameter_id, count in counts.items():
            print(f"{indent}parameter {parameter_id}: {count} cases")

    for station_id, station_df in repeated_cases_by_station.items():
        print("\n" + "=" * 70)
        print(f"STATION: {station_id}")
        print("=" * 70)

        station_total_cases = 0

        for event_aquocode in valid_aquocodes:
            df_aquocode = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df_aquocode)
            n_unique_parameters = df_aquocode[target_parameter_col].nunique()
            station_total_cases += n_cases

            print(f"\n  AQUOCODE {event_aquocode}")
            print(f"  Cases: {n_cases} | Unique parameters: {n_unique_parameters}")

            if n_cases == 0:
                print("  No rows")
                continue

            # Aquocode level
            print("  Summary (all hoedanigheid_code):")
            print_parameter_summary(df_aquocode, indent="    ")

            # Hoedanigheid level
            hoedanigheid_groups = (
                df_aquocode
                .groupby(hoedanigheid_col, dropna=False)
                .size()
                .sort_values(ascending=False)
            )

            print("  By hoedanigheid_code:")
            for hoedanigheid_code in hoedanigheid_groups.index:
                df_h = df_aquocode[df_aquocode[hoedanigheid_col] == hoedanigheid_code]
                n_cases_h = len(df_h)
                n_unique_h = df_h[target_parameter_col].nunique()

                print(
                    f"    - {hoedanigheid_code}: "
                    f"{n_cases_h} cases, {n_unique_h} unique parameters"
                )
                print_parameter_summary(df_h, indent="      ")

        print(f"\n  STATION TOTAL: {station_total_cases} cases")
        grand_total_cases += station_total_cases

    print("\n" + "=" * 70)
    print(f"GRAND TOTAL: {grand_total_cases} cases")
    print("=" * 70)

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    columns_to_keep = [
        target_loc_col,
        "grootheid_code",
        target_parameter_col,
        "hoedanigheid_code",
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        # normalize to calendar day (ignore time part)
        station_df_duplicates["event_day"] = pd.to_datetime(
            station_df_duplicates["eventdatum"], errors="coerce"
        ).dt.normalize()

        def same_day_duplicate_info(group):
            # count how many measurements per day within this case
            day_counts = group["event_day"].value_counts(dropna=True)

            # days with more than 1 measurement
            has_same_day = (day_counts > 1).any()

            # unique-parameter ids that appear on a duplicated day
            # (for this grouping, it will usually be the same id, but kept generic)
            if has_same_day:
                ids = sorted(group[target_parameter_col].dropna().astype(str).unique())
                ids_text = ", ".join(ids)
            else:
                ids_text = ""

            return pd.Series(
                {
                    "n_measurements": group["eventdatum"].size,
                    "n_unique_dates": group["eventdatum"].nunique(),
                    "first_date": group["eventdatum"].min(),
                    "last_date": group["eventdatum"].max(),
                    "same_day_measurement": "yes" if has_same_day else "no",
                    "same_day_parameter_ids": ids_text,
                }
            )
        
        value_cols = ["eventdatum", "event_day", target_parameter_col]

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)[value_cols]
            .apply(same_day_duplicate_info)
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
    show_only_duplicates=True,
    top_n=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    hoedanigheid_col = "hoedanigheid_code"
    same_day_col = "same_day_measurement"  # expects values like "yes"/"no"

    grand_total_cases = 0
    grand_total_same_day_yes = 0

    def print_parameter_summary(df, indent="      "):
        counts = df[target_parameter_col].value_counts(ascending=False)

        if show_only_duplicates:
            counts = counts[counts > 1]

        if top_n is not None:
            counts = counts.head(top_n)

        if len(counts) == 0:
            print(f"{indent}No duplicate parameters")
            return

        for parameter_id, count in counts.items():
            print(f"{indent}parameter {parameter_id}: {count} cases")

    for station_id, station_df in repeated_cases_by_station.items():
        print("\n" + "=" * 70)
        print(f"STATION: {station_id}")
        print("=" * 70)

        station_total_cases = 0
        station_total_same_day_yes = 0

        for event_aquocode in valid_aquocodes:
            df_aquocode = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df_aquocode)
            n_unique_parameters = df_aquocode[target_parameter_col].nunique()
            station_total_cases += n_cases

            # Count rows/cases flagged as yes for same-day measurements
            n_same_day_yes = (
                df_aquocode[same_day_col]
                .astype(str)
                .str.lower()
                .eq("yes")
                .sum()
            )
            station_total_same_day_yes += n_same_day_yes

            print(f"\n  AQUOCODE {event_aquocode}")
            print(f"  Cases: {n_cases} | Unique parameters: {n_unique_parameters}")
            print(f"  Cases with same-day measurements (yes): {n_same_day_yes}")

            if n_cases == 0:
                print("  No rows")
                continue

            # Aquocode level
            print("  Summary (all hoedanigheid_code):")
            print_parameter_summary(df_aquocode, indent="    ")

            # Hoedanigheid level
            hoedanigheid_groups = (
                df_aquocode
                .groupby(hoedanigheid_col, dropna=False)
                .size()
                .sort_values(ascending=False)
            )

            print("  By hoedanigheid_code:")
            for hoedanigheid_code in hoedanigheid_groups.index:
                df_h = df_aquocode[df_aquocode[hoedanigheid_col] == hoedanigheid_code]
                n_cases_h = len(df_h)
                n_unique_h = df_h[target_parameter_col].nunique()
                n_same_day_yes_h = (
                    df_h[same_day_col]
                    .astype(str)
                    .str.lower()
                    .eq("yes")
                    .sum()
                )

                print(
                    f"    - {hoedanigheid_code}: "
                    f"{n_cases_h} cases, {n_unique_h} unique parameters, "
                    f"{n_same_day_yes_h} with same-day measurements (yes)"
                )
                print_parameter_summary(df_h, indent="      ")

        print(f"\n  STATION TOTAL: {station_total_cases} cases")
        print(f"  STATION TOTAL with same-day measurements (yes): {station_total_same_day_yes} cases")

        grand_total_cases += station_total_cases
        grand_total_same_day_yes += station_total_same_day_yes

    print("\n" + "=" * 70)
    print(f"GRAND TOTAL: {grand_total_cases} cases")
    print(f"GRAND TOTAL with same-day measurements (yes): {grand_total_same_day_yes} cases")
    print("=" * 70)


Missing location data: 0 rows
Shape of df_data after mapping location: (33403, 45)
Shape of df_mapping: (79, 2)
Shape of df_data after mapping location: (33403, 46)
Shape of df_data after dropping rows without parameter: (7692, 46)
Shape of filtered DataFrame (excluding BER): (7692, 46)
Shape of filtered DataFrame (bemonsteringshoogte_code == -100): (5622, 46)
Shape of filtered DataFrame (valid event_aquocode): (5622, 46)
Unieke identificatie gemeten parameter
1369.0    156
1379.0    156
1382.0    156
1386.0    156
1392.0    156
         ... 
5347.0     52
5978.0     52
5979.0     52
7073.0     32
1289.0      3
Name: count, Length: 79, dtype: int64
Shape of unique_values_after_merge: (79,)
No identical rows found.
Created 5 station DataFrames
Created 5 repeated-case tables


### Running tasks

In [ ]:
df_data = mapping_location(df_data, df_mapping_location)
df_data = mapping_substances(df_data, df_mapping_substances)
df_data = drop_rows_without_parameter(df_data)
filtered_df = filter_data(df_data)
unique_values_after_merge = get_count_unique_values(filtered_df)
check_duplicates(filtered_df)
dfs_by_station = split_by_station(filtered_df)
repeated_cases_by_station = get_repeated_cases_by_station(dfs_by_station)
print_parameter_counts_by_event_aquocode(repeated_cases_by_station)


STATION: NL89_SCHAARVODDL

  AQUOCODE 0
  Cases: 78 | Unique parameters: 71
  Cases with same-day measurements (yes): 0
  Summary (all hoedanigheid_code):
    parameter 1386.0: 2 cases
    parameter 1382.0: 2 cases
    parameter 1387.0: 2 cases
    parameter 1379.0: 2 cases
    parameter 1392.0: 2 cases
    parameter 1369.0: 2 cases
    parameter 1388.0: 2 cases
  By hoedanigheid_code:
    - NVT: 68 cases, 68 unique parameters, 0 with same-day measurements (yes)
      No duplicate parameters
    - nf: 8 cases, 8 unique parameters, 0 with same-day measurements (yes)
      No duplicate parameters
    - Nnf: 1 cases, 1 unique parameters, 0 with same-day measurements (yes)
      No duplicate parameters
    - Pnf: 1 cases, 1 unique parameters, 0 with same-day measurements (yes)
      No duplicate parameters

  AQUOCODE 3
  Cases: 49 | Unique parameters: 43
  Cases with same-day measurements (yes): 0
  Summary (all hoedanigheid_code):
    parameter 1392.0: 2 cases
    parameter 1388.0: 2 ca

In [ ]:
repeated_cases_by_station["NL89_SCHAARVODDL"]

,Identitication unique de la station,grootheid_code,Unieke identificatie gemeten parameter,hoedanigheid_code,event_aquocode,eenheid_code,n_measurements,n_unique_dates,first_date,last_date,same_day_measurement,same_day_parameter_ids
0,NL89_SCHAARVODDL,CONCTTE,1337.0,nf,0,mg/l,26,26,2024-01-10,2024-12-17,no,
1,NL89_SCHAARVODDL,CONCTTE,1339.0,Nnf,0,mg/l,26,26,2024-01-10,2024-12-17,no,
2,NL89_SCHAARVODDL,CONCTTE,1433.0,Pnf,0,mg/l,26,26,2024-01-10,2024-12-17,no,
3,NL89_SCHAARVODDL,CONCTTE,1387.0,NVT,0,ug/l,26,26,2024-01-10,2024-12-17,no,
4,NL89_SCHAARVODDL,CONCTTE,1382.0,nf,0,ug/l,23,23,2024-01-10,2024-12-17,no,
...,...,...,...,...,...,...,...,...,...,...,...,...
122,NL89_SCHAARVODDL,CONCTTE,1629.0,NVT,3,ug/l,1,1,2024-12-17,2024-12-17,no,
123,NL89_SCHAARVODDL,CONCTTE,5347.0,NVT,3,ug/l,1,1,2024-08-08,2024-08-08,no,
124,NL89_SCHAARVODDL,CONCTTE,5979.0,NVT,3,ug/l,1,1,2024-08-08,2024-08-08,no,
125,NL89_SCHAARVODDL,CONCTTE,6616.0,NVT,3,ug/l,1,1,2024-09-25,2024-09-25,no,


In [ ]:
# I'll report by substance, then location, and then code